## Introduction
Cancer treatment is not equally effective for every patient. Tumours that look similar clinically can respond very differently to the same anticancer drug. One reason for this is the variety of genomic alterations present in the cells. Some of these alterations affect the biological pathways a drug targets and/or the mechanisms by which a cancer cell can survive drug treatment. If we can characterize the molecular features of a cancer cell, perhaps we can predit which drugs are likely to work on that cell. The Genomics of Drug Sensitivity in Cancer (GDSC) project has generated a large dataset combining:

- cancer cell lines
- genomic/molecular characteristics of those cell lines
- measurement of their responses to anticancer compounds

This makes the GDSC well suited to investigating the relationship between genomic features and drug response. 

Because the genomic feature space is enormous, Machine Learning (ML) provides methods for efficiently learning the relationships between genomic features and drug response. Importantly, it is uncertain whether genomic information actually contains enough signal to predict drug response.

**Question:** Can anticancer drug response be predicted from genomic features using machine-learning models trained on GDSC data, and which genomic features contribute most to predictive performance?

This question can be broken down into sevearal sub-questions:
1. **Predictability:** Is anticancer drug response predictable from genomic features?
2. **Model Performance:** Which machine-learning models provice the most useful predictive performance? Which models are most efficient? Is there an intersection between performance and efficiency?
3. **Generalization:** Does the predictive performance of the models generalize to unseen data?
4. **Biological Interpretation:** Which genomic features contribute most to predictive performance? Can we interpret the models to understand the biological mechanisms underlying drug response?
5. **Drug Specificity:** Are there specific drugs for which genomic features are particularly predictive of response? Conversely, are there drugs for which genomic features provide little predictive power?

**Null Hypothesis:** Genomic features do not contain enough information to predict anticancer drug response, with the methods explored.

**Alternative Hypothesis:** Genomic features exist that can provide predictive information about anticancer drug response, and the machine-learning models exploured can be trained to leverage this information effectively.

## Data Ingestion
### Overview

The data ingestion phase established a reproducible pipeline for acquiring and combining the datasets required to investigate whether anticancer drug response can be predicted from genomic features.

The analysis uses GDSC release 8.4, released July 24, 2022, for drug-response measurements and cell-line metadata. Gene-expression features are obtained separately from the COSMIC Cell Lines Project, version 104, using the GRCh38 release. The two sources use different identifiers, so an explicit mapping procedure was required before the datasets could be joined.

The goal of this phase was to acquire the original source data without modifying it, validate its structure, and construct a reproducible representation suitable for subsequent preprocessing.

### GDSC response data

The primary drug-response data were obtained from the official Sanger CancerRxGene GDSC release 8.4. The release contains two fitted single-agent response datasets:

- GDSC1 fitted dose-response data
- GDSC2 fitted dose-response data

Both datasets were downloaded and retained rather than selecting one at the ingestion stage. They were concatenated into a single response table while retaining the DATASET field so that the origin of each observation remains identifiable.

The response data contain, among other fields:

- `COSMIC_ID`
- `CELL_LINE_NAME`
- `DRUG_NAME`
- `AUC`
- `LN_IC50`
- `DATASET`

Both AUC and LN_IC50 were retained at this stage. Selection of the final response variable is a modelling decision and therefore belongs to the preprocessing/analysis phase rather than data acquisition.

### GDSC cell-line metadata

The GDSC release also provides Cell_Lines_Details.xlsx. This file was downloaded alongside the response data and used to attach biological metadata to the response observations.

The relevant metadata include:

- GDSC tissue of origin
- secondary tissue descriptor
- TCGA cancer type
- COSMIC identifier
- sample name
- availability indicators for WES, CNA, gene expression, and methylation

The tissue-of-origin field is particularly important because the research question is intended to support cancer-location-specific analyses rather than treating all cancer cell lines as a single homogeneous population.

The metadata are joined to the response data using the numeric COSMIC_ID supplied by GDSC.

### Gene-expression data

The GDSC release metadata indicate whether gene-expression data are available for each cell line, but the actual genomic feature matrix is not contained in the GDSC response files. Consequently, a second data source was required.

The selected genomic modality is gene expression from the COSMIC Cell Lines Project.

The downloaded COSMIC product is:

- Cell Lines Project Complete Gene Expression
- Version 104
- GRCh38
- Affymetrix Human Genome U219 Array

The data are supplied as a compressed TSV file contained within a TAR archive:

`CellLinesProject_CompleteGeneExpression_v104_GRCh38.tar`

containing:

`CellLinesProject_CompleteGeneExpression_v104_GRCh38.tsv.gz`

The expression file is initially in long format, with each row representing a cell-line/gene combination. The relevant fields are:

- COSMIC_SAMPLE_ID
- SAMPLE_NAME
- COSMIC_GENE_ID
- GENE_SYMBOL
- REGULATION
- Z_SCORE
- COSMIC_STUDY_ID

For this project, Z_SCORE is used as the expression measurement.

## Handling the COSMIC download

Unlike the historical GDSC release files, the current COSMIC download does not provide a permanent public URL. COSMIC generates a user-specific, time-limited signed URL.

Because the URL contains credentials and an expiration timestamp, it should not be committed to the repository. Instead, the current URL is supplied through an environment variable:

`COSMIC_LINK`

The ingestion code loads this value from `.env` using `python-dotenv`.

This approach allows the data-acquisition code to remain reproducible without embedding a user`s private, temporary COSMIC download URL in source control.

A complication encountered during development was that the signed URL could return HTTP 403 errors when it had expired. Refreshing the COSMIC download URL and placing the new value in .env resolved the problem. The already-downloaded archive is subsequently reused, so the signed URL is not required every time the dataset is loaded.

## Archive extraction

The COSMIC download is distributed as a TAR archive rather than directly as the expression TSV. The ingestion code therefore:

1. Obtains the signed URL from COSMIC_LINK.
2. Downloads the TAR archive to data/raw.
3. Inspects the archive contents.
4. Locates the expected expression file.
5. Extracts CellLinesProject_CompleteGeneExpression_v104_GRCh38.tsv.gz.
6. Retains the original compressed expression file for subsequent loading.

The archive and extracted expression file are treated as raw inputs rather than modified analytical datasets.

## Mapping GDSC to COSMIC

A major challenge was that GDSC and COSMIC do not use the same identifier for the cell lines.

GDSC provides a numeric `COSMIC_ID`, whereas the COSMIC expression dataset identifies samples using `COSMIC_SAMPLE_ID`, such as `COSS905985`.

The COSMIC sample file establishes the relationship between the COSMIC sample identifier and `SAMPLE_NAME`. The GDSC cell-line metadata also contains `Sample Name`.

Therefore, the mapping was established through the cell-line sample name:

`GDSC COSMIC_ID → GDSC Sample Name → COSMIC SAMPLE_NAME → COSMIC_SAMPLE_ID`

This was preferable to attempting to infer the relationship from the cell-line names in the drug-response table alone.

An empirical validation of this mapping was performed before incorporating expression data.

The comparison found:

- 1,002 GDSC sample names
- 1,020 COSMIC sample names
- 999 matching names
- 99.7% of GDSC names matched to COSMIC
- no COSMIC sample names mapped to multiple COSMIC sample IDs

Three GDSC names did not have a corresponding COSMIC sample name:

- `GT3TKB`
- `Hep 3B2_1-7`
- `TOTAL:`

`TOTAL:` was identified as a summary row rather than a biological cell line and was explicitly excluded from the mapping process.

The remaining unmatched names are retained as unmatched rather than being assigned an inferred identity.

## Expression data duplication

During integration, an apparent uniqueness problem was discovered in the COSMIC expression data.

The raw expression dataset contains approximately 17.5 million rows. Initial inspection showed approximately 1.86 million duplicate rows, indicating that the assumption that every `COSMIC_SAMPLE_ID`/`GENE_SYMBOL` combination was unique was incorrect.

Further investigation showed that the duplication was associated primarily with two COSMIC studies:

- COSU3
- COSU619

The duplication was not simply a matter of identical repeated records. Within `COSU619`, 11,688 sample/gene pairs occurred twice. Of those pairs, 9,814 had identical Z-scores, while 1,874 had slightly different Z-scores.

The two genes involved were:

- `ATXN7`
- `TMSB15B`

The differences were extremely small. For example, the largest observed differences were approximately 0.013 Z-score units.

This investigation was important because it demonstrated that blindly using `pivot()` with an assumed unique sample/gene key would be inappropriate. It also established that the duplicate records need to be understood as part of the COSMIC data structure rather than treated automatically as errors.

The ingestion phase therefore does not silently discard these observations. The duplicate structure is documented and must be resolved explicitly during preprocessing before constructing the final feature matrix.

## Long-to-wide transformation

The COSMIC expression source is distributed in long format:

`sample × gene → expression value`

Machine-learning models require a feature matrix in which each observation has a set of genomic features. The eventual representation therefore needs to be transformed into:

`cell line → gene 1, gene 2, gene 3, ...`

with the corresponding Z-score for each gene.

The ingestion code contains a preparation step for this transformation, but the duplicate observations discovered above mean that the final aggregation rule should be established during preprocessing rather than hidden inside the raw data-ingestion step.

This separation is intentional: ingestion should preserve the source data and expose structural problems, while preprocessing should document and apply the analytical decisions used to resolve them.

## Reproducibility and validation

The ingestion pipeline records the GDSC release information and the COSMIC expression product used. A manifest is written to the raw-data directory containing:

- GDSC release number
- GDSC release date
- source location
- source filenames
- COSMIC expression product
- COSMIC version
- genome build
- expected archive and expression filenames

Validation functions check that the expected GDSC response files, cell-line metadata, and COSMIC expression file exist and contain the required columns.

The pipeline is designed so that existing downloaded files are reused rather than downloaded repeatedly. This is particularly important for the COSMIC dataset because its download URL is temporary and user-specific.

## Result of the ingestion phase

At the end of data ingestion, the project has three principal raw data resources:

1. GDSC1 drug-response data
2. GDSC2 drug-response data
3. GDSC cell-line metadata
4. COSMIC v104 gene-expression data

The response data and metadata can be joined using `COSMIC_ID`. The expression data can be linked to the GDSC cell lines through the validated `SAMPLE_NAME` mapping.

No biological observations have yet been removed because of missing genomic features, no expression values have been imputed, no genes have been selected or filtered, and no machine-learning train/test split has been performed.

These decisions are deliberately deferred to the preprocessing stage.

The key output of ingestion is therefore *a validated, reproducible connection between drug-response observations, cancer/tissue metadata, and COSMIC gene-expression measurements*, while preserving enough information about the original sources and identifier mappings to make the subsequent analysis auditable.


We are investigating the merge of the GDSC set with the COSMIC set, it isn't as straightforward as it initially seemed.

### Initial dataset characterization

The following summaries describe the loaded release before any preprocessing or analytical filtering. Tissue and cancer labels are retained as metadata; no cancer type is selected at this stage.

### Genomic availability and quality checks

The response files contain no genomic feature matrix. `Cell_Lines_Details.xlsx` records whether WES, CNA, gene expression, and methylation assays are available for each model; the actual feature files and their identifiers must be selected and joined in a later research-design step. The checks below report raw-data limitations without removing observations.

### Data-to-Preprocessing checkpoint

This analysis uses official Sanger CancerRxGene GDSC release 8.4 (24 July 2022), combining GDSC1 and GDSC2 fitted single-agent response files with `Cell_Lines_Details.xlsx`. Each observation represents a cell-line/drug response record and retains COSMIC model ID, tissue of origin, and cancer metadata. AUC and LN_IC50 are available and complete in the loaded release; neither is selected as the final target yet.

The release metadata reports WES, CNA, gene-expression, and methylation availability flags, but does not include the genomic feature matrices themselves. Cancer type is missing for a substantial subset of observations, while tissue labels are retained. Before preprocessing, the project still needs to choose a genomic modality and source, define eligibility and missingness rules, select a response measure, and decide how cancer types will be compared. No observations have been imputed, filtered, normalized, or split at this checkpoint.

### Reproducible preprocessing interface

The preprocessing implementation lives in `gdsc.preprocessing`, so the notebook does not duplicate hidden cleaning steps. It requires a selected genomic matrix with one verified shared cell-line identifier (normally `COSMIC_ID`). The downloaded release inspected above does not supply that matrix: its availability flags only indicate whether an assay exists for a model. Therefore executing a real join here would falsely imply that a genomic modality has been selected.

Once a feature matrix is obtained, call `preprocess_gdsc` with the documented study thresholds. The returned `X` will contain genomic columns only; `y` will be the explicit AUC or LN_IC50 choice; and tissue, cancer type, drug, and identifiers will remain in `metadata`. The function records every row/feature count in `prepared.info`. It does not fit an imputer or scaler. Any later transformer must be fit on the training split only.

The next cell is deliberately a readiness check, not a call to `preprocess_gdsc`: no genomic feature matrix is currently available. This checkpoint intentionally stops before a train/test split and model training.

### Reproducible preprocessing interface

The preprocessing implementation lives in `gdsc.preprocessing`, so the notebook does not duplicate hidden cleaning steps. It requires a selected genomic matrix with one verified shared cell-line identifier (normally `COSMIC_ID`). The downloaded release inspected above does not supply that matrix: its availability flags only indicate whether an assay exists for a model. Therefore executing a real join here would falsely imply that a genomic modality has been selected.

Once a feature matrix is obtained, call `preprocess_gdsc` with the documented study thresholds. The returned `X` will contain genomic columns only; `y` will be the explicit AUC or LN_IC50 choice; and tissue, cancer type, drug, and identifiers will remain in `metadata`. The function records every row/feature count in `prepared.info`. It does not fit an imputer or scaler. Any later transformer must be fit on the training split only.

No preprocessing call is shown here because no genomic feature matrix is currently available. This checkpoint intentionally stops before a train/test split and model training.

### Memory-safe expression access

A previous ingestion attempt merged the complete COSMIC expression matrix (about 17,000 genes) onto all GDSC drug-response observations (about 575,000 rows). That would create a dense response-row-by-gene table, and it failed with a memory allocation request of roughly 72.8 GiB. This was an architecture problem, not a reason to discard observations or genes.

The replacement is a feature-store workflow. GDSC response records and metadata remain a lightweight table. COSMIC expression is de-duplicated using the documented arithmetic mean for repeated sample/gene Z-scores and cached once as long-format Parquet in `data/processed/cosmic_expression.parquet`. During preprocessing, the project will first select the cohort, cell lines, and genes, then call `load_expression_features` for only that subset. Only that bounded subset may be merged into a modelling dataset.

This preserves the raw files, avoids repeating the expensive preparation step, and prevents a future full-expression/full-response merge from recreating the memory failure.

### Analysis pipeline

```text
Raw source data
│
├── GDSC1 / GDSC2 drug-response files
├── GDSC cell-line metadata
└── COSMIC v104 gene-expression data
        │
        ▼
Data ingestion and preparation
│
├── Validate source files and schemas
├── Standardize identifiers and metadata
├── Map GDSC cell lines to COSMIC samples
├── Resolve documented duplicate expression records
└── Cache expression data as Parquet
        │
        ▼
Analytical source data
│
├── GDSC response + metadata table
└── COSMIC expression feature store
        │
        ▼
Preprocessing
│
├── Select tissue / cancer cohort
├── Select response variable
├── Define drug eligibility
├── Define cell-line eligibility
├── Query relevant expression features
├── Handle missing data
├── Filter / select genes
├── Construct feature matrix X and target y
├── Split training / validation / test data
└── Apply scaling or transformations as appropriate
        │
        ▼
Modeling
│
├── Train baseline models
├── Train candidate machine-learning models
├── Tune model hyperparameters
└── Evaluate predictive performance
        │
        ▼
Interpretation
│
├── Compare model performance
├── Estimate feature importance
├── Identify genomic features associated with prediction
└── Assess biological and methodological implications


The important boundary is that the **Parquet feature store is the output of ingestion/preparation, not the output of ML preprocessing**.

## Preprocessing decisions implemented in the module

Preprocessing now uses a drug-specific observation unit: one response per eligible cell line for one selected drug. The module first selects a tissue, summarizes drug eligibility within that cohort, maps only the retained cell lines to COSMIC, and requests only the chosen genes from the Parquet store. This prevents the former all-responses by all-genes memory failure from returning.

The resulting `X` contains expression Z-scores only; `y` is an explicit AUC or LN_IC50 choice; cell-line identifiers, drug, tissue, cancer type, and dataset remain separate metadata. Missingness and variance are reported before filtering. Any imputation, variance filtering, or optional scaling is represented by an unfitted scikit-learn pipeline and must be fit on training data only. Splits are grouped by `COSMIC_ID`, preventing one cell line from leaking into more than one split.

In [1]:
from gdsc.data import load_gdsc

gdsc = load_gdsc(
    data_dir="../data/raw",
    include_metadata=True,
)

gdsc.head()

/home/ajharris/Projects/gdsc-project/venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,COSMIC_ID,CELL_LINE_NAME,SANGER_MODEL_ID,TCGA_DESC,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,...,Copy Number Alterations (CNA),Gene Expression,Methylation,Drug\nResponse,TISSUE_OF_ORIGIN,TISSUE_DESCRIPTOR_2,CANCER_TYPE,Microsatellite \ninstability Status (MSI),Screen Medium,Growth Properties
0,GDSC1,361,17635802,684057,ES5,SIDM00263,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
1,GDSC1,361,17636176,684059,ES7,SIDM00269,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
2,GDSC1,361,17636568,684062,EW-11,SIDM00203,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent
3,GDSC1,361,17636912,684072,SK-ES-1,SIDM01111,UNCLASSIFIED,1,Erlotinib,EGFR,...,Y,Y,Y,Y,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Semi-Adherent
4,GDSC1,361,17637300,687448,COLO-829,SIDM00909,SKCM,1,Erlotinib,EGFR,...,Y,Y,Y,Y,skin,melanoma,SKCM,MSS/MSI-L,R,Adherent


## Preprocessing Question 1: what tissues give us enough cell lines and drug-response coverage to work with?

### Selecting a tissue cohort

Before constructing the machine-learning dataset, the analysis must define a tissue of origin. Because model performance depends on having enough independent cell lines with measured drug responses, tissue selection should be informed by the coverage available in GDSC rather than chosen arbitrarily.

The following summary compares each tissue by its number of unique cell lines, tested drugs, and total drug-response observations. In particular, the number of unique cell lines is important because cell lines represent the independent biological samples available for training and evaluating drug-specific models.

In [2]:
from gdsc.preprocessing import summarize_tissues

# Reusable summary; missing tissue labels are shown rather than silently removed.
tissue_summary = summarize_tissues(gdsc)
tissue_summary

,CELL_LINES,DRUGS,RESPONSE_OBSERVATIONS,cell_lines,observations
TISSUE_OF_ORIGIN,,,,,
lung_NSCLC,108,542,64499,108,64499
urogenital_system,104,542,61288,104,61288
leukemia,84,542,50008,84,50008
aero_dig_tract,77,542,45354,77,45354
lymphoma,69,542,40948,69,40948
lung_SCLC,63,542,33558,63,33558
skin,58,542,33253,58,33253
nervous_system,55,542,32794,55,32794
breast,52,542,31021,52,31021


### Tissue selection -> Lung

The tissue-coverage analysis identified lung_NSCLC as the largest available cohort, containing 108 unique cell lines, 542 tested drugs, and 64,499 drug-response observations. Because the subsequent analysis will construct drug-specific models, the number of unique cell lines is more important than the total number of response records: each selected drug will be represented by only the subset of cell lines in which it was tested.

For the initial analysis, lung_NSCLC will therefore be used as the tissue cohort. Selecting the largest available cohort maximizes the number of independent biological samples available for model development and evaluation while establishing a workflow that can subsequently be repeated for other tissues.

### Drug-response coverage within the NSCLC cohort

We now select `lung_NSCLC` using the reusable preprocessing function. Drug coverage is then summarized within this cohort only. `N_CELL_LINES`, not raw response rows, is the primary descriptive sample-size statistic because a future drug-specific model has one row per cell line. Duplicate drug/cell-line records are diagnosed below but are not averaged or removed at this stage.


In [3]:
import importlib
import gdsc.preprocessing as preprocessing

# Reload the local module so this cell reflects edits made during notebook development.
preprocessing = importlib.reload(preprocessing)

# The module performs all reusable selection and diagnostic calculations.
# This notebook only presents the documented results for the chosen cohort.
coverage_report = preprocessing.analyze_cohort_drug_coverage(
    gdsc,
    tissue_of_origin="lung_NSCLC",
    thresholds=(20, 30, 40, 50, 60, 75, 90, 100),
)
drug_summary = coverage_report["drug_summary"]
identity_diagnostics = coverage_report["identity_diagnostics"]
duplicate_diagnostics = coverage_report["duplicate_diagnostics"]
coverage_diagnostics = coverage_report["coverage_diagnostics"]
eligibility_decision_table = coverage_report["eligibility_decision_table"]

print(coverage_diagnostics["distribution"])
display(eligibility_decision_table)
display(drug_summary.reset_index(drop=True).head(20))
print({key: len(value) for key, value in identity_diagnostics.items()})
{key: value for key, value in duplicate_diagnostics.items() if key != "duplicated_pairs"}


count    542.000000
mean      95.201107
std       21.824840
min        5.000000
25%       98.250000
50%      104.000000
75%      108.000000
max      108.000000
Name: N_CELL_LINES, dtype: float64


,minimum_unique_cell_lines,eligible_drugs
0,20,537
1,30,510
2,40,510
3,50,508
4,60,501
5,75,501
6,90,411
7,100,381


,DRUG_NAME,N_OBSERVATIONS,N_CELL_LINES,N_AUC_AVAILABLE,AUC_MISSING_FRACTION,N_LN_IC50_AVAILABLE,LN_IC50_MISSING_FRACTION,DATASETS,PUTATIVE_TARGET,DRUG_ID,observations,cell_lines
0,Selumetinib,394,108,394,0.0,394,0.0,"(GDSC1, GDSC2)","(MEK1, MEK2,)","(1062, 1498, 1736)",394,108
1,AZD4547,322,108,322,0.0,322,0.0,"(GDSC1, GDSC2)","(FGFR1, FGFR2, FGFR3, FGRF1, FGFR2, FGFR3)","(1135, 1497, 1786)",322,108
2,PLX-4720,321,108,321,0.0,321,0.0,"(GDSC1, GDSC2)","(BRAF,)","(1036, 1371)",321,108
3,AZD7762,320,108,320,0.0,320,0.0,"(GDSC1, GDSC2)","(CHEK1, CHEK2,)","(1022, 1402)",320,108
4,Olaparib,319,108,319,0.0,319,0.0,"(GDSC1, GDSC2)","(PARP1, PARP2,)","(1017, 1495)",319,108
5,Pictilisib,319,108,319,0.0,319,0.0,"(GDSC1, GDSC2)","(PI3K (class 1),)","(1058, 1527)",319,108
6,Afatinib,318,108,318,0.0,318,0.0,"(GDSC1, GDSC2)","(EGFR, ERBB2,)","(1032, 1377)",318,108
7,SN-38,318,108,318,0.0,318,0.0,"(GDSC1, GDSC2)","(TOP1,)","(1490, 1494)",318,108
8,Avagacestat,315,108,315,0.0,315,0.0,"(GDSC1, GDSC2)","(Amyloid beta20, Amyloid beta40,)","(205, 1072)",315,108
9,Gemcitabine,315,108,315,0.0,315,0.0,"(GDSC1, GDSC2)","(Pyrimidine antimetabolite,)","(135, 1190, 1393)",315,108


{'name_to_multiple_ids': 71, 'id_to_multiple_names': 0}


{'n_drug_cell_line_records': 64499,
 'n_unique_drug_cell_line_pairs': 57786,
 'n_duplicated_drug_cell_line_pairs': 6713,
 'max_records_per_pair': 2}

## Defining the first reproducible experiment

Drug-specific models need enough independent cell lines for development and evaluation. We therefore require at least 75 unique lung_NSCLC cell lines per drug. The initial candidate is selected strictly by coverage, then by lowest DRUG_ID in a tie. AUC is the primary continuous response: it summarizes the fitted dose-response curve, while LN_IC50 remains in the source data for a later sensitivity analysis.

GDSC1 and GDSC2 measurements are separate screens, not values to average. We choose the screen with the most usable AUC measurements for the selected drug (GDSC1 wins only an exact tie), then verify that every stable DRUG_ID/COSMIC_ID pair is unique within that source.

In [4]:
experiment_response = preprocessing.build_initial_response_cohort(
    gdsc, tissue_of_origin="lung_NSCLC", min_unique_cell_lines=75, response_metric="AUC"
)
selected_drug = experiment_response["selected_drug"]
response_cohort = experiment_response["response_cohort"]
display(experiment_response["eligibility"]["eligible_drugs"].reset_index(drop=True).head(20))
display(selected_drug[["DRUG_ID", "DRUG_NAME", "N_CELL_LINES", "N_OBSERVATIONS", "DATASETS", "PUTATIVE_TARGET"]].to_frame().T)
display(experiment_response["dataset_coverage"])
print({
    "selected_dataset": experiment_response["selected_dataset"],
    "screen_specific_drug_id": experiment_response["selected_drug_id"],
    "response_rows": len(response_cohort),
    "unique_cell_lines": response_cohort["COSMIC_ID"].nunique(),
    "missing_auc_rows_excluded": experiment_response["n_excluded_response_rows"],
})


,DRUG_NAME,N_OBSERVATIONS,N_CELL_LINES,N_AUC_AVAILABLE,AUC_MISSING_FRACTION,N_LN_IC50_AVAILABLE,LN_IC50_MISSING_FRACTION,DATASETS,PUTATIVE_TARGET,DRUG_ID,observations,cell_lines
0,Selumetinib,394,108,394,0.0,394,0.0,"(GDSC1, GDSC2)","(MEK1, MEK2,)","(1062, 1498, 1736)",394,108
1,AZD4547,322,108,322,0.0,322,0.0,"(GDSC1, GDSC2)","(FGFR1, FGFR2, FGFR3, FGRF1, FGFR2, FGFR3)","(1135, 1497, 1786)",322,108
2,PLX-4720,321,108,321,0.0,321,0.0,"(GDSC1, GDSC2)","(BRAF,)","(1036, 1371)",321,108
3,AZD7762,320,108,320,0.0,320,0.0,"(GDSC1, GDSC2)","(CHEK1, CHEK2,)","(1022, 1402)",320,108
4,Olaparib,319,108,319,0.0,319,0.0,"(GDSC1, GDSC2)","(PARP1, PARP2,)","(1017, 1495)",319,108
5,Pictilisib,319,108,319,0.0,319,0.0,"(GDSC1, GDSC2)","(PI3K (class 1),)","(1058, 1527)",319,108
6,Afatinib,318,108,318,0.0,318,0.0,"(GDSC1, GDSC2)","(EGFR, ERBB2,)","(1032, 1377)",318,108
7,SN-38,318,108,318,0.0,318,0.0,"(GDSC1, GDSC2)","(TOP1,)","(1490, 1494)",318,108
8,Avagacestat,315,108,315,0.0,315,0.0,"(GDSC1, GDSC2)","(Amyloid beta20, Amyloid beta40,)","(205, 1072)",315,108
9,Gemcitabine,315,108,315,0.0,315,0.0,"(GDSC1, GDSC2)","(Pyrimidine antimetabolite,)","(135, 1190, 1393)",315,108


,DRUG_ID,DRUG_NAME,N_CELL_LINES,N_OBSERVATIONS,DATASETS,PUTATIVE_TARGET
97,1,Erlotinib,108,129,"(GDSC1, GDSC2)","(EGFR,)"


,DATASET,N_RESPONSE_ROWS,N_CELL_LINES,N_USABLE_RESPONSE_CELL_LINES,N_MISSING_RESPONSE
0,GDSC1,21,21,21,0
1,GDSC2,108,108,108,0


{'selected_dataset': 'GDSC2', 'screen_specific_drug_id': 1168, 'response_rows': 108, 'unique_cell_lines': 108, 'missing_auc_rows_excluded': 0}


### Why the files have different roles

Raw GDSC CSV files, the metadata workbook, and compressed COSMIC TSV files in `data/raw/` are preserved source inputs. The TSV is too large to load as one dataframe, so cache construction reads it in chunks and uses a temporary SQLite database only to aggregate duplicate sample/gene Z-scores. That temporary database is deleted afterwards. The durable result is a long-format Parquet file in `data/processed/`; Parquet is used because it can retrieve only the selected COSMIC sample IDs and is column sorted as opposed to a row sorted delimited text file. Only this selected response cohort is pivoted to a cell-line-by-gene matrix. No all-responses-by-all-genes table is created.

### Why Parquet is used for expression data

COSMIC gene-expression data are stored as Parquet because the expression dataset is a large, read-mostly numerical feature matrix that is repeatedly subsetted for analysis. Parquet provides efficient compressed storage and allows the pipeline to retrieve only the expression data needed for the selected cell-line cohort. SQLite would also support efficient targeted queries, but its row-oriented relational structure is less natural for the wide feature matrices ultimately required for machine learning. Importantly, the main memory optimization comes from selecting the experimental cohort before retrieving expression features, rather than from Parquet itself.

In [5]:
from gdsc.cosmic import build_expression_cache

# Run once to create/reuse the preferred long-format feature store.
# It reads the raw COSMIC TSV in chunks, uses temporary SQLite aggregation,
# and writes data/processed/cosmic_expression.parquet (not a response-by-gene table).
processed_expression_cache = build_expression_cache(
    "../data", progress=print, total_source_rows=17_467_776
)
processed_expression_cache


Reusing existing expression cache: ../data/processed/cosmic_expression.parquet


PosixPath('../data/processed/cosmic_expression.parquet')

### Targeted expression dataset and quality review

Only after fixing the tissue, drug, screen, and response do we retrieve expression. The next call maps the final response cohort to COSMIC, asks Parquet for those samples only, and pivots only that bounded result into a cell-line-by-gene matrix. It applies no target-driven feature selection. The documented 20% gene-missingness rule and zero-variance removal are target-independent; imputation is deferred until after splitting.

In [6]:
experiment_dataset, experiment_response = preprocessing.build_initial_experiment_dataset(
    gdsc, data_dir="../data", tissue_of_origin="lung_NSCLC",
    min_unique_cell_lines=75, response_metric="AUC",
    max_gene_missing_fraction=0.20,
)
missingness = experiment_dataset.diagnostics["missingness_before_filtering"]
preprocessing_report = {
    "response_cell_lines": experiment_dataset.diagnostics["n_response_cell_lines"],
    "mapped_cosmic_samples": experiment_dataset.diagnostics["n_mapped_to_cosmic"],
    "unmatched_cell_lines": experiment_dataset.diagnostics["n_unmatched"],
    "expression_available_cell_lines": experiment_dataset.diagnostics["n_with_expression"],
    "excluded_no_expression": len(experiment_dataset.diagnostics["excluded_no_expression_ids"]),
    "genes_before_filtering": missingness["n_genes"],
    "genes_after_filtering": experiment_dataset.diagnostics["n_genes_after_filtering"],
    "all_missing_genes": len(missingness["all_missing_genes"]),
    "total_missing_values": missingness["total_missing"],
    "overall_missing_fraction": missingness["overall_missing_fraction"],
    "X_shape": experiment_dataset.X.shape,
    "y_shape": experiment_dataset.y.shape,
    "metadata_shape": experiment_dataset.metadata.shape,
}
assert len(experiment_dataset.X) == len(experiment_dataset.y) == len(experiment_dataset.metadata)
assert experiment_dataset.X.index.equals(experiment_dataset.y.index)
assert experiment_dataset.X.index.equals(experiment_dataset.metadata.index)
preprocessing_report


{'response_cell_lines': 108,
 'mapped_cosmic_samples': 108,
 'unmatched_cell_lines': 0,
 'expression_available_cell_lines': 106,
 'excluded_no_expression': 2,
 'genes_before_filtering': 16980,
 'genes_after_filtering': 16980,
 'all_missing_genes': 0,
 'total_missing_values': 0,
 'overall_missing_fraction': 0.0,
 'X_shape': (106, 16980),
 'y_shape': (106,),
 'metadata_shape': (106, 9)}

### Leakage-safe modeling handoff

Splitting is grouped by `COSMIC_ID`, so no biological cell line can appear in more than one partition. The median imputer and variance filter learn feature statistics and therefore are fit on training expression only. COSMIC values are already Z-scores, so scaling is disabled. Validation and test data are transformed with training-derived statistics only; this notebook does not fit a predictive model.

In [7]:
splits = preprocessing.split_by_cell_line(
    experiment_dataset, test_fraction=0.20, validation_fraction=0.20, random_state=42
)
train, validation, test = splits["train"], splits["validation"], splits["test"]
fitted_preprocessor = preprocessing.build_preprocessor(imputation="median", scaling=False)
X_train = fitted_preprocessor.fit_transform(train.X)
X_val = fitted_preprocessor.transform(validation.X)
X_test = fitted_preprocessor.transform(test.X)
y_train, y_val, y_test = train.y, validation.y, test.y
metadata_train, metadata_val, metadata_test = train.metadata, validation.metadata, test.metadata
split_ids = [set(part.metadata["COSMIC_ID"]) for part in (train, validation, test)]
assert not (split_ids[0] & split_ids[1] or split_ids[0] & split_ids[2] or split_ids[1] & split_ids[2])
modeling_handoff = {
    "X_train": X_train.shape, "X_val": X_val.shape, "X_test": X_test.shape,
    "y_train": y_train.shape, "y_val": y_val.shape, "y_test": y_test.shape,
    "metadata_train": metadata_train.shape, "metadata_val": metadata_val.shape,
    "metadata_test": metadata_test.shape,
    "genes_after_training_only_variance_filter": X_train.shape[1],
    "scaling": "disabled: COSMIC values are already Z-scores",
}
modeling_handoff


{'X_train': (63, 16980),
 'X_val': (21, 16980),
 'X_test': (22, 16980),
 'y_train': (63,),
 'y_val': (21,),
 'y_test': (22,),
 'metadata_train': (63, 9),
 'metadata_val': (21, 9),
 'metadata_test': (22, 9),
 'genes_after_training_only_variance_filter': 16980,
 'scaling': 'disabled: COSMIC values are already Z-scores'}

### Drug eligibility decision table

A drug is *eligible* at a stated threshold when it has response measurements from at least that many **unique cell lines**. The table is calculated by `filter_eligible_drugs`; raw response rows are not treated as independent biological samples. For the approved initial experiment, the threshold is **75 unique lung_NSCLC cell lines**. It remains a parameter so the workflow can be repeated under a different documented criterion.

The first experiment uses a deterministic availability rule rather than expected biology or predictive results: among eligible drugs, `select_initial_drug` chooses the greatest cell-line coverage and breaks an exact tie by the lowest `DRUG_ID`. AUC is the primary target; LN_IC50 remains available for later sensitivity analysis. GDSC1 and GDSC2 measurements are not averaged. Instead, `select_response_dataset` chooses the source screen with the greatest usable AUC coverage (GDSC1 only wins an exact tie) and raises an error if duplicate stable drug-ID/cell-line records remain within that screen.
## Preprocessing status

The data-ingestion phase is complete. GDSC release 8.4 drug-response data and cell-line metadata can be loaded successfully, and COSMIC gene-expression data have been integrated through a separate, memory-safe feature store. The ingestion pipeline handles GDSC/COSMIC sample mapping, duplicate expression measurements, caching, and targeted expression access without constructing an impractically large response-by-gene table.

The preprocessing phase has now begun. The first step was to examine the number of independent cell lines available for each tissue of origin. lung_NSCLC was selected for the initial analysis because it contains the largest cohort in the dataset, with 108 unique cell lines, 542 drugs, and 64,499 response observations. The workflow remains parameterized so that the same analysis can later be repeated for other tissues.

Drug eligibility is now parameterized and shown as a threshold decision table. Although 542 drugs are represented, not every drug was necessarily tested against all 108 cell lines. Because the eventual models will be drug-specific, the number of unique cell lines with a response measurement for each drug determines the effective sample size for that model.

The first reproducible experiment is now defined: 75 unique cell lines for eligibility, deterministic coverage-based drug selection, AUC as the response metric, and one GDSC screen selected by usable cell-line coverage. This resolves cross-screen duplication by choosing a consistent experimental source, never by averaging. Within-screen duplicate stable drug-ID/cell-line records remain a hard error.

The implemented path now constructs the resolved response cohort first, maps only those cell lines to COSMIC, and queries only those expression records from the Parquet feature store. The raw CSV/TSV/XLSX files remain source inputs in `data/raw/`. SQLite is used only as a temporary on-disk aggregation workspace while converting the large COSMIC TSV in chunks; it is deleted after the cache is built. The durable long-format Parquet cache is stored in `data/processed/` and is queried only after cohort selection. A bounded cell-line-by-gene matrix is then produced; the prohibited full response-row-by-gene merge is never created.

Current milestone: **preprocessing is complete and verified on the real initial experiment.** Erlotinib from GDSC2 with AUC produces a targeted 106-cell-line, 16,980-gene matrix after COSMIC mapping and target-independent filtering. Grouped splitting produces 63 training, 21 validation, and 22 test cell lines. Median imputation and variance filtering are fitted only on training features; scaling is disabled because COSMIC values are already Z-scores. No predictive model is fit in preprocessing.

## Modeling: validation-only baselines

The first modeling question is whether genomic expression improves on predicting the training-set mean AUC. Ridge and Elastic Net are regularized linear models appropriate when genes greatly outnumber training cell lines. Their fixed parameters are baselines, not a hyperparameter search. Models fit training data and are compared on validation data only; the held-out test set remains untouched.

In [8]:
import pandas as pd
from gdsc.evaluation import fit_and_evaluate_validation
from gdsc.models import build_dummy_regressor, build_elastic_net_model, build_ridge_model

baseline_models = {
    "Mean baseline": build_dummy_regressor(),
    "Ridge (alpha=1.0)": build_ridge_model(alpha=1.0),
    "Elastic Net (alpha=1.0, l1_ratio=0.5)": build_elastic_net_model(),
}
validation_results = []
for name, model in baseline_models.items():
    _, metrics = fit_and_evaluate_validation(model, X_train, y_train, X_val, y_val)
    validation_results.append({"MODEL": name, **metrics.__dict__})

print({"train_cell_lines": len(y_train), "validation_cell_lines": len(y_val),
       "held_out_test_cell_lines": len(y_test), "genes": X_train.shape[1],
       "features_per_training_cell_line": X_train.shape[1] / len(y_train)})
display(pd.DataFrame(validation_results).set_index("MODEL"))
# X_test and y_test are deliberately not used in this baseline-comparison cell.


{'train_cell_lines': 63, 'validation_cell_lines': 21, 'held_out_test_cell_lines': 22, 'genes': 16980, 'features_per_training_cell_line': 269.5238095238095}


,mae,rmse,pearson,spearman,r2
MODEL,,,,,
Mean baseline,0.068166,0.088695,NaN,NaN,-0.010572
Ridge (alpha=1.0),0.066239,0.081292,0.445491,0.354545,0.151076
"Elastic Net (alpha=1.0, l1_ratio=0.5)",0.068166,0.088695,NaN,NaN,-0.010572


### Candidate model development

Baseline models establish whether predictive signal is present. We now make a small, conservative comparison between tuned regularized linear models and one nonlinear Random Forest. Hyperparameters are selected by three-fold cross-validation **within training data only**, using RMSE as the primary error metric. Each selected estimator is evaluated once on validation; the held-out test set remains untouched.

Modeling approaches

The modeling phase begins by asking a simple question: do the COSMIC gene-expression features contain enough information to predict drug response better than a trivial baseline? Because the dataset contains many more genomic features than independent cell lines, the analysis starts with simple, regularized models before considering more flexible nonlinear methods.

Mean-response baseline

The first model is a DummyRegressor that predicts the mean AUC observed in the training set for every validation sample.

This model uses no genomic information. Its purpose is to establish a minimum reference level of predictive performance. A genomic model is only useful if it can improve meaningfully on this baseline.

The mean-response baseline is evaluated using the same validation metrics as the genomic models so that improvements can be measured directly.

Ridge regression

Ridge regression is used as the primary linear genomic baseline.

Ordinary least-squares regression is not appropriate for this problem because the number of gene-expression features is much larger than the number of available cell lines. In such a high-dimensional setting, an unrestricted linear model can overfit severely and may not have a unique or stable solution.

Ridge regression addresses this problem by adding an L2 regularization penalty to the regression coefficients. This discourages very large coefficient values while still allowing all genes to contribute to the prediction.

Conceptually, Ridge balances two objectives:

fitting the observed drug-response values;
keeping the overall magnitude of the model coefficients small.

The strength of regularization is controlled by the parameter alpha. Larger values impose stronger shrinkage, while smaller values make the model more similar to ordinary linear regression.

Because many gene-expression features may be correlated, Ridge is a useful starting point: correlated genes can retain shared predictive contribution rather than forcing the model to select only one of them.

Elastic Net

Elastic Net provides a second regularized linear approach.

It combines:

L1 regularization, which can shrink some coefficients exactly to zero;
L2 regularization, which stabilizes coefficient estimates in the presence of correlated predictors.

This makes Elastic Net useful for testing whether predictive performance can be maintained using a smaller subset of genomic features.

Two main parameters control the model:

alpha, which determines the overall strength of regularization;
l1_ratio, which determines the balance between L1 and L2 penalties.

A low l1_ratio behaves more like Ridge regression, while a high l1_ratio produces stronger feature sparsity.

At this stage, Elastic Net is treated as a predictive comparison rather than a definitive feature-selection method. Non-zero coefficients should not yet be interpreted as biologically important genes.

Random Forest

Random Forest provides a nonlinear comparison to the regularized linear models.

A Random Forest combines predictions from many decision trees trained on different subsets of observations and features. Unlike Ridge and Elastic Net, it can represent nonlinear relationships and interactions among genes without requiring those relationships to be specified in advance.

This is useful because drug response may depend on combinations of genomic characteristics that cannot be represented by a purely linear model.

However, Random Forest must be used cautiously in this dataset. The number of independent cell lines is relatively small compared with the number of gene-expression features. Flexible nonlinear models can therefore overfit easily.

For this reason, the Random Forest is treated as a limited candidate model rather than the starting point of the analysis. Its complexity is controlled using parameters such as tree depth, minimum leaf size, and the number of features considered at each split.

Hyperparameter selection

Model hyperparameters are selected using cross-validation performed entirely within the training set.

For example, candidate Ridge alpha values can be evaluated using training-only cross-validation. The best-performing configuration is then refit using the full training set and evaluated on the validation set.

The same principle applies to Elastic Net and Random Forest.

The data flow is therefore:

Training set
    ↓
internal cross-validation
    ↓
select hyperparameters
    ↓
refit selected configuration on full training set
    ↓
evaluate on validation set

The validation set is used to compare modeling approaches, while the held-out test set remains untouched.

Evaluation metrics

Several complementary regression metrics are used because no single metric fully describes predictive performance.

Mean Absolute Error (MAE) measures the average absolute difference between predicted and observed AUC values. Lower values indicate better predictions.

Root Mean Squared Error (RMSE) also measures prediction error but penalizes large errors more strongly than MAE.

Pearson correlation measures the linear relationship between predicted and observed drug response.

Spearman correlation evaluates whether the model correctly ranks cell lines from relatively more sensitive to relatively more resistant, regardless of whether the relationship is perfectly linear.

R² may also be reported as a measure of explained variation, although it is interpreted cautiously because negative values are possible on unseen data.

Together, these metrics provide information about both numerical prediction accuracy and the ability of the model to preserve relative response patterns.

Model comparison strategy

The models are compared in increasing order of complexity:

Mean-response baseline
        ↓
Ridge regression
        ↓
Elastic Net
        ↓
Random Forest

This ordering is intentional.

If Ridge regression substantially outperforms the mean baseline, that provides evidence that the genomic features contain predictive information even under a relatively simple linear model.

If Elastic Net performs similarly while using fewer effective features, this may suggest that a smaller subset of expression features contains much of the predictive signal.

If Random Forest provides a meaningful additional improvement, that may indicate nonlinear relationships or interactions among expression features.

More complicated models are therefore justified only when they provide a clear improvement over simpler alternatives.

Test-set isolation

The held-out test set is not used during model development.

It is not used to:

select model families;
choose hyperparameters;
select genes;
modify preprocessing;
decide which validation metric to emphasize.

Once the modeling strategy and hyperparameters have been selected using training and validation data, the strategy is locked. The test set is then evaluated once to provide an estimate of final generalization performance.

This separation reduces the risk of indirectly overfitting the analysis to the test data.

Interpretation comes later

The modeling phase initially focuses on predictive performance rather than biological interpretation.

Coefficient magnitudes, feature importance, SHAP values, gene rankings, and pathway-level interpretation are deferred until a predictive modeling strategy has been selected and evaluated.

This separation is important because a feature that is useful for prediction is not automatically causal or biologically important, and feature interpretation can itself introduce additional analytical choices.

The modeling workflow is therefore:

Establish trivial baseline
        ↓
Establish regularized linear baselines
        ↓
Evaluate a limited nonlinear candidate
        ↓
Select model using validation data
        ↓
Lock modeling strategy
        ↓
Evaluate held-out test set
        ↓
Interpret genomic features

The immediate goal is not to identify the most important genes, but first to determine whether gene-expression data provide reproducible predictive signal for anticancer drug response.

## Modeling approaches

### Tuned Ridge Regression

Ridge regression tries to predict drug response by combining information from all of the measured genes, with each gene receiving a numerical weight. Because there are thousands of genes but relatively few cell lines, an unrestricted model could easily fit random quirks in the training data rather than patterns that generalize. Ridge addresses this by penalizing excessively large weights, effectively encouraging the model to make more conservative predictions. The tuned version tests several strengths of this penalty using only the training data and selects the value that performs most consistently. Ridge is particularly useful here because many genes may carry overlapping or correlated information.

### Tuned Elastic Net

Elastic Net is another linear model, but unlike Ridge it can effectively remove genes from the prediction by shrinking some of their weights all the way to zero. It combines two forms of regularization: one that stabilizes the model when many genes contain related information, and another that encourages the model to rely on a smaller subset of features. The tuned version determines both how strongly the model should be regularized and the balance between these two effects using the training data. This provides an interesting comparison with Ridge: Ridge asks whether predictive information is distributed across many genes, while Elastic Net tests whether similar performance can be achieved with a more selective set of genes.

### Bounded Random Forest

Random Forest takes a very different approach. Instead of assuming that drug response can be represented as a weighted combination of gene-expression values, it builds many decision trees, each of which learns a series of rules from the data. Their predictions are then combined. This allows Random Forest to capture nonlinear relationships and interactions—for example, situations where the importance of one gene may depend on the expression of another. However, with thousands of genes and relatively few cell lines, unrestricted trees could easily memorize the training data. The forest is therefore bounded, meaning its complexity is deliberately limited through parameters such as tree depth and minimum leaf size. This provides a controlled test of whether nonlinear relationships improve prediction without giving the model unlimited flexibility.

### Why compare all three?

The three models test progressively different ideas about how gene expression might relate to drug response:

Ridge asks whether predictive information is broadly distributed across many genes. Elastic Net asks whether a more selective subset of genes can make useful predictions. Bounded Random Forest asks whether nonlinear relationships and interactions provide information that the linear models cannot capture.

Comparing them against the same simple mean-response baseline helps determine not only whether genomic information is predictive, but also what level of model complexity is actually justified by the available data.

In [9]:
from gdsc.models import (
    build_elastic_net_model, build_random_forest_model, build_ridge_model,
    tune_training_only,
)
from gdsc.evaluation import evaluate_regression

candidate_specs = {
    "Tuned Ridge": (build_ridge_model(), {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}),
    "Tuned Elastic Net": (build_elastic_net_model(), {"alpha": [0.01, 0.1, 1.0], "l1_ratio": [0.1, 0.5, 0.9]}),
    # A single shallow, 50-tree configuration is a bounded nonlinear comparison.
    # A large forest grid is impractical with 16,980 genes and 63 training lines.
    "Random Forest (bounded)": (build_random_forest_model(n_estimators=50, max_depth=5, min_samples_leaf=2, max_features=0.1), {"max_depth": [5]}),
}
candidate_results = []
for name, (estimator, grid) in candidate_specs.items():
    fitted, cv_result = tune_training_only(estimator, grid, X_train, y_train, cv=3)
    metrics = evaluate_regression(y_val, fitted.predict(X_val))
    candidate_results.append({"MODEL": name, **cv_result, **metrics.__dict__})

validation_candidates = pd.DataFrame(candidate_results).set_index("MODEL").sort_values("rmse")
validation_candidates  # VALIDATION RESULTS ONLY: X_test/y_test are not used here.


,parameters,cv_rmse_mean,cv_rmse_std,mae,rmse,pearson,spearman,r2
MODEL,,,,,,,,
Tuned Ridge,{'alpha': 100.0},0.102637,0.010035,0.066219,0.081280,0.445106,0.354545,0.151336
Random Forest (bounded),{'max_depth': 5},0.109097,0.023194,0.065792,0.083055,0.358506,0.224675,0.113860
Tuned Elastic Net,"{'alpha': 1.0, 'l1_ratio': 0.5}",0.108449,0.029802,0.068166,0.088695,NaN,NaN,-0.010572


### Locked-model test evaluation

**Lock recorded before test evaluation:** tuned Ridge with `alpha=100.0`, median imputation and training-only variance filtering, 16,980 genes, fitted on the 63-cell-line training partition only. Ridge was selected by lowest validation RMSE (0.081280) with the most stable training-CV RMSE (0.102637 ± 0.010035). The test set was excluded from preprocessing, CV, and model-family selection. Opening it below is the single final generalization evaluation for this experiment: no hyperparameters, features, preprocessing, or model family may be changed afterward.

In [10]:
from gdsc.evaluation import evaluate_locked_model
from gdsc.models import build_dummy_regressor, build_ridge_model

# No tuning occurs here. Both estimators fit X_train/y_train only.
_, dummy_test_metrics, dummy_predictions = evaluate_locked_model(
    build_dummy_regressor(), X_train, y_train, X_test, y_test, metadata_test
)
_, ridge_test_metrics, ridge_predictions = evaluate_locked_model(
    build_ridge_model(alpha=100.0), X_train, y_train, X_test, y_test, metadata_test
)
held_out_test_results = pd.DataFrame([
    {"MODEL": "Mean-response baseline", **dummy_test_metrics.__dict__},
    {"MODEL": "Locked Ridge (alpha=100.0)", **ridge_test_metrics.__dict__},
]).set_index("MODEL")
held_out_test_results  # HELD-OUT TEST RESULTS: do not retune after this cell.


,mae,rmse,pearson,spearman,r2
MODEL,,,,,
Mean-response baseline,0.054069,0.061048,NaN,NaN,-0.448617
Locked Ridge (alpha=100.0),0.050805,0.060515,0.354313,0.247883,-0.423419


## Feature Interpretation

Absolute signed Ridge coefficients rank predictive contribution on the unchanged COSMIC Z-score matrix. Positive/negative signs associate higher expression with higher/lower *predicted* AUC conditional on this model. They are predictive associations, not evidence of biological causation. Training-only bootstrap refits assess stability without tuning, changing features, or using test responses.

In [11]:
from gdsc.interpretation import ridge_bootstrap_stability, ridge_coefficients
from gdsc.models import build_ridge_model
locked_ridge = build_ridge_model(alpha=100.0).fit(X_train, y_train)
feature_ranking = ridge_coefficients(locked_ridge, train.X.columns)
feature_stability = ridge_bootstrap_stability(lambda: build_ridge_model(alpha=100.0), X_train, y_train, train.X.columns, n_resamples=100, random_state=42)
feature_ranking.head(20).merge(feature_stability, on="GENE_SYMBOL")


,GENE_SYMBOL,COEFFICIENT,ABS_COEFFICIENT,SIGN,COEFFICIENT_MEAN,COEFFICIENT_MEDIAN,COEFFICIENT_STD,POSITIVE_FRACTION,NEGATIVE_FRACTION
0,YTHDF3,0.000285,0.000285,1,0.000190,0.000205,0.000067,1.00,0.00
1,BAG5,-0.000247,0.000247,-1,-0.000167,-0.000201,0.000080,0.03,0.97
2,CYB5B,0.000235,0.000235,1,0.000151,0.000199,0.000085,0.98,0.02
3,SETD3,0.000225,0.000225,1,0.000139,0.000180,0.000079,0.98,0.02
4,TMEM11,-0.000223,0.000223,-1,-0.000148,-0.000148,0.000042,0.00,1.00
5,WDR13,0.000221,0.000221,1,0.000135,0.000164,0.000078,0.92,0.08
6,MSTN,0.000220,0.000220,1,0.000139,0.000157,0.000057,0.98,0.02
7,GSTA3,-0.000220,0.000220,-1,-0.000146,-0.000148,0.000047,0.00,1.00
8,DHRS7B,-0.000216,0.000216,-1,-0.000148,-0.000149,0.000033,0.00,1.00
9,RTF1,-0.000215,0.000215,-1,-0.000130,-0.000145,0.000042,0.00,1.00


### How to read the feature-interpretation table

This table describes feature-interpretation results from the locked Ridge model. The final-model `COEFFICIENT` shows how each gene contributes to **predicted AUC** in that fitted model. The resampling columns show how stable that relationship is when the training data are perturbed and the same locked model is refit. These are predictive associations, not evidence that a gene causes drug response. Higher predicted AUC should be interpreted according to the response definition established earlier in the notebook.

- `COEFFICIENT` gives direction: positive means higher expression is associated with higher predicted AUC in the fitted model; negative means higher expression is associated with lower predicted AUC. `SIGN` is simply `+1` or `-1` for that direction.
- `ABS_COEFFICIENT` ignores direction. The table is ranked by this value, so larger values indicate stronger influence **within this particular Ridge model**. They do not identify the most biologically important gene.
- `COEFFICIENT_MEAN` and `COEFFICIENT_MEDIAN` summarize the same coefficient across repeated training-set resamples, showing whether the final fitted value is typical. `COEFFICIENT_STD` measures how much it changes: lower values indicate more stable magnitude, while higher values indicate more variability.
- `POSITIVE_FRACTION` and `NEGATIVE_FRACTION` show how often a coefficient kept each direction. A value of 1.00 means that direction appeared in 100% of resampled fits; values near 0.5 indicate unstable direction.

For example, **YTHDF3** is a strong positive and highly stable feature (`POSITIVE_FRACTION=1.00`), whereas **TMEM11** is a strong negative and highly stable feature (`NEGATIVE_FRACTION=1.00`). **DHRS7B** is negative with especially low coefficient variability (`COEFFICIENT_STD=0.000033`), while **WDR13** is mostly positive but less perfectly stable (`POSITIVE_FRACTION=0.92`). Many top-ranked genes have highly stable directions across resamples, which is encouraging for interpretation. However, Ridge distributes predictive weight across correlated genes, so a high coefficient does not mean that one gene alone drives a biological effect.

The next useful question is whether the top-ranked genes are highly correlated with one another or with broader groups of genes, because Ridge can distribute weight across correlated features.

### Correlation among top predictive features

The following training-only calculation examines the top 20 fixed Ridge features. Correlated expression can make an individual coefficient harder to interpret: Ridge may distribute predictive weight across related genes. These correlations are descriptive; no genes are removed, no model is refit, and no test responses are used.

In [12]:
from gdsc.interpretation import top_feature_correlations
top_gene_names = feature_ranking.head(20)["GENE_SYMBOL"].tolist()
top_feature_correlation_pairs = top_feature_correlations(train.X, top_gene_names, top_n=20)
top_feature_correlation_pairs.head(20)


,GENE_A,GENE_B,CORRELATION,ABS_CORRELATION
0,TMEM11,DHRS7B,0.752273,0.752273
1,TMEM11,AKAP10,0.670153,0.670153
2,CYB5B,DDT,0.489348,0.489348
3,BAG5,ISCA2,0.489120,0.489120
4,DHRS7B,AKAP10,0.481228,0.481228
5,DDT,NDUFB9,0.481068,0.481068
6,DHRS7B,SRR,0.446882,0.446882
7,CYB5B,SFR1,0.434718,0.434718
8,SETD3,DDTL,0.384198,0.384198
9,BAG5,CYB5B,-0.367261,0.367261


### Feature interpretation summary

The locked Ridge model was interpreted without changing the model, preprocessing, feature set, or held-out test evaluation. We ranked all 16,980 expression features by the absolute magnitude of their signed Ridge coefficients and displayed a fixed top-20 summary. Because the inputs are COSMIC expression Z-scores, coefficient magnitudes are comparable within this fitted linear model: positive coefficients are associated with higher predicted AUC and negative coefficients with lower predicted AUC. This is predictive association only, not evidence that a gene causes drug response; the meaning of higher AUC should be read using the response definition established earlier.

To assess robustness, the same locked Ridge configuration (`alpha=100.0`) was repeatedly refit on bootstrap resamples of the training data only. The reported coefficient mean, median, standard deviation, and positive/negative fractions describe how consistently each feature's magnitude and direction persisted. Many top-ranked features have highly stable directions, which supports cautious interpretation of the fitted model, but it does not establish biological mechanism or statistical significance.

Finally, correlations among the fixed top 20 training-expression features were examined without removing or changing any genes. Correlated expression matters because Ridge can distribute predictive weight across related genes: an individual large coefficient does not imply that one gene alone drives the prediction. The next stage, if pursued, should keep this correlation structure in view while investigating broader biological context; it should not revisit the locked predictive-performance result.

## Biological Context: Targeted Literature Review

The computational portion of the primary experiment has identified gene-expression features that contribute most strongly to the locked model's predictions. The next step is to investigate whether existing biological research provides context for these predictive associations.

This is a **targeted, model-driven literature review**, rather than a comprehensive review of the drug, cancer type, or every gene identified by the model. The purpose is to compare the computational findings with existing biological knowledge while keeping the two sources of evidence separate.

The literature review should address three questions:

1. **What is already known about the drug and its mechanism?**
2. **Is there existing evidence connecting the model's highest-ranked and most stable genes to the drug, its molecular target or pathways, drug response, or the relevant cancer context?**
3. **Do several of the predictive features point toward a common biological process that could provide context for the model's predictions?**

### Separating model findings from biological evidence

Three different types of statements must be kept distinct throughout the review.

**Model finding:**  
A result produced by the present computational analysis. For example, a gene may have a large positive Ridge coefficient and retain the same coefficient direction across nearly all training-set resamples.

**Existing evidence:**  
A finding reported independently in the scientific literature. This might establish that the gene participates in a pathway targeted by the drug, has previously been associated with drug response, or has a known role in the relevant cancer type.

**Interpretation or hypothesis:**  
A possible explanation suggested by considering the model result together with existing evidence. Such an interpretation can motivate further investigation, but it does not establish a causal relationship.

A gene's importance to the predictive model is therefore **not evidence that the gene causes drug sensitivity or resistance**. Gene-expression features are correlated, and Ridge regression can distribute predictive information among related genes. A highly ranked gene may consequently represent a broader cellular state or expression program rather than an independent biological mechanism.

### Literature review strategy

The review will begin with the drug itself. Before interpreting individual genes, I will establish its known molecular target(s), mechanism of action, major associated pathways, and documented mechanisms of sensitivity or resistance, particularly where these are relevant to the tissue being studied.

I will then examine the highest-ranked features identified during model interpretation. Priority will be given to genes that combine a relatively large model coefficient with strong stability across training-set resampling.

For each gene, I will look for evidence at progressively broader levels:

- a **direct relationship with the selected drug or response to that drug**;
- a relationship with the **drug's molecular target or target pathway**;
- evidence concerning **drug sensitivity or resistance mechanisms**;
- a relationship with the **selected cancer/tissue context**;
- involvement in a biological process plausibly relevant to the drug's action;
- or **no clear relevant relationship in the available literature**.

The final category is important. The goal is not to construct a biological explanation for every predictive feature. If no convincing relationship can be identified, that result should be recorded rather than replaced with a speculative explanation.

### Recording the evidence

For each investigated gene, I will maintain a structured record containing:

| Field | What to record |
|---|---|
| Gene | Gene symbol |
| Model result | Rank, coefficient direction, and stability |
| Evidence category | Direct, target/pathway-related, cancer-context-related, indirect/plausible, or no clear connection |
| Biological finding | Brief description of what the literature actually reports |
| Relevance | Why the finding may or may not provide context for the model result |
| Source | PMID and/or DOI |
| Notes | Important limitations or uncertainties |

Where possible, specific mechanistic claims will be supported by **primary peer-reviewed research**. Review articles may be used to establish broader drug, pathway, or disease context. References should be recorded as the review proceeds so that every biological claim in the final analysis can be traced back to its source.

### Suggested search progression

For each important feature, searches should proceed from specific to broad. For a hypothetical gene `GENE1` and drug `DRUG`, useful searches would include:

    "GENE1" "DRUG"
    "GENE1" drug resistance
    "GENE1" drug sensitivity
    "GENE1" "[drug target]"
    "GENE1" "[target pathway]"
    "GENE1" NSCLC
    "GENE1" lung cancer

A lack of results from the most specific searches should not automatically be interpreted as evidence that no relationship exists. Broader pathway and cancer-context searches may reveal an indirect connection. At the same time, increasingly broad results should be treated as progressively weaker evidence for explaining the model result.

### Evidence categories

To keep the interpretation conservative and consistent, literature findings will be classified as:

**Direct** — Published evidence directly connects the gene to the selected drug or response to that drug.

**Target/pathway related** — Evidence connects the gene to the drug's molecular target or a clearly relevant pathway, but not necessarily to response to the specific drug.

**Cancer-context related** — Evidence connects the gene to the selected cancer type or a relevant cancer phenotype without establishing a drug-specific relationship.

**Indirect/plausible** — A possible mechanistic relationship can be proposed from existing research, but it requires substantial inference.

**No clear connection identified** — The targeted search did not identify convincing evidence connecting the feature to the drug or relevant mechanism.

These categories describe the **strength and proximity of the existing biological context**, not the statistical importance of the feature.

### What would constitute an interesting result?

Several outcomes would be informative.

If a highly ranked and stable feature has already been directly associated with response to the same drug, the model may be recovering a previously recognized biological relationship.

If several top features independently connect to the drug's target or a common pathway, the model may be detecting a broader biological state associated with response.

If a highly stable predictive feature has no obvious connection in the literature, it remains an interesting model finding, but should be treated as a candidate for further investigation rather than evidence of a new mechanism.

Conversely, if the top features have little coherent biological relationship to the drug, that is also important. It may indicate that prediction depends on indirect expression patterns, correlated cellular states, or signals that are difficult to assign to individual genes.

### Stopping rule

This review is deliberately bounded. The objective is to provide biological context for the **predefined top-ranked features**, not to search indefinitely until every gene can be given a plausible explanation.

Once the selected features have been systematically reviewed, their evidence categories recorded, and any well-supported common themes identified, the biological-context phase will be considered complete.

The next computational step will be a **response-metric sensitivity analysis using LN_IC50**. The purpose of that experiment will be to determine whether the predictive performance and feature-level conclusions obtained using AUC remain similar when drug response is represented by a different metric.